Một chiến lược mua: Khi giá dự đoán tăng so với giá hiện tại, Momentum dương (điều này cho thấy xu hướng giá đang tăng), và RSI dưới 70 (điều này cho thấy rằng cổ phiếu không bị mua quá mức).

Một chiến lược bán: Khi giá dự đoán giảm so với giá hiện tại, Momentum âm (điều này cho thấy xu hướng giá đang giảm), và RSI trên 30 (điều này cho thấy cổ phiếu không bị bán quá mức).

# Load du lieu len

In [1]:
import sys
sys.path.append('../Common')

import CommonBinance
from datetime import datetime, timedelta

symbol = 'ETHUSDT'
from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
timeframe = '1m' # 1m

data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)

data

,Datetime,Open,High,Low,Close,Volume
0,2025-10-22 00:00:00,3873.04,3873.04,3862.16,3864.99,1188.6477
1,2025-10-22 00:01:00,3864.98,3868.76,3862.67,3868.76,332.1784
2,2025-10-22 00:02:00,3868.75,3872.91,3868.65,3872.91,340.9633
3,2025-10-22 00:03:00,3872.91,3877.00,3872.90,3877.00,209.3692
4,2025-10-22 00:04:00,3877.00,3878.70,3874.81,3877.14,339.9579
...,...,...,...,...,...,...
816,2025-10-22 13:36:00,3836.81,3844.15,3836.81,3839.97,840.6678
817,2025-10-22 13:37:00,3839.96,3851.83,3835.22,3847.25,994.9095
818,2025-10-22 13:38:00,3847.25,3855.59,3847.25,3854.40,481.8656
819,2025-10-22 13:39:00,3854.35,3859.63,3854.35,3858.38,829.6720


# Tim tham so p, d, q toi uu: Tim 1 lan thoi

In [16]:
# import itertools
# import statsmodels.api as sm
# import pandas as pd

# # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# # data.set_index('Datetime', inplace=True) da dat o tren

# # Xác định khoảng giá trị cho tham số p, d, q
# p = d = q = range(0, 6)
# pdq = list(itertools.product(p, d, q))

# best_aic = float("inf")
# best_pdq = None
# best_model = None

# for param in pdq:
#     try:
#         model = sm.tsa.ARIMA(data['Close'], order=param)
#         results = model.fit()
#         if results.aic < best_aic:
#             best_aic = results.aic
#             best_pdq = param
#             best_model = results
#     except:
#         continue

# print(f'Best ARIMA{best_pdq} AIC:{best_aic}')


# Tinh toan chien luoc

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import sys

# Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
data.set_index('Datetime', inplace=True)

# Tính Momentum
data['Momentum'] = data['Close'].diff()

# Tính RSI
delta = data['Close'].diff()
up, down = delta.clip(lower=0), delta.clip(upper=0).abs()
roll_up, roll_down = up.rolling(14).mean(), down.rolling(14).mean()
RS = roll_up / roll_down
data['RSI'] = 100.0 - (100.0 / (1.0 + RS))

# Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
model = ARIMA(data['Close'], order=(5, 1, 5))
model_fit = model.fit()

# Dự đoán giá
data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

# Xác định tín hiệu mua và bán
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > -1) & (data['RSI'] < 100))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 1201) & (data['RSI'] > 50))

# Trực quan hóa
import plotly.graph_objects as go

# Tạo biểu đồ cơ bản với giá đóng cửa
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))

# Thêm dữ liệu giá dự đoán vào biểu đồ
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close', line=dict(dash='dot', color='red')))

# Thêm điểm mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))

# Thêm điểm bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Tùy chỉnh layout
fig.update_layout(title='Trading Signals with ARIMA, Momentum, and RSI', xaxis_title='Date', yaxis_title='Price', hovermode='x')

# Hiển thị biểu đồ
fig.show()

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [3]:
data

,Open,High,Low,Close,Volume,Momentum,RSI,Predicted_Close,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,
2025-10-22 00:00:00,3873.04,3873.04,3862.16,3864.99,1188.6477,NaN,NaN,0.000000,False,False
2025-10-22 00:01:00,3864.98,3868.76,3862.67,3868.76,332.1784,3.77,NaN,3864.991882,False,False
2025-10-22 00:02:00,3868.75,3872.91,3868.65,3872.91,340.9633,4.15,NaN,3868.924053,False,False
2025-10-22 00:03:00,3872.91,3877.00,3872.90,3877.00,209.3692,4.09,NaN,3873.012368,False,False
2025-10-22 00:04:00,3877.00,3878.70,3874.81,3877.14,339.9579,0.14,NaN,3876.858788,False,False
...,...,...,...,...,...,...,...,...,...,...
2025-10-22 13:36:00,3836.81,3844.15,3836.81,3839.97,840.6678,3.17,33.675663,3836.630084,False,False
2025-10-22 13:37:00,3839.96,3851.83,3835.22,3847.25,994.9095,7.28,44.164429,3841.312692,False,False
2025-10-22 13:38:00,3847.25,3855.59,3847.25,3854.40,481.8656,7.15,50.917015,3846.867239,False,True


In [19]:
import plotly.graph_objects as go

# Giả sử DataFrame của bạn tên là `data` và có chỉ mục datetime
fig = go.Figure()

# Thêm đường biểu diễn giá đóng cửa và giá dự đoán
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

# Thêm điểm cho tín hiệu mua và bán
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], 
                         mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], 
                         mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Thêm đường cho Momentum và RSI với trục y phụ
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], name='Momentum', yaxis='y2'))
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], name='RSI', yaxis='y3'))

# Tùy chỉnh layout cho các trục y phụ
fig.update_layout(
    title='Trading Signals with ARIMA, Momentum, and RSI',
    xaxis_title='Date',
    yaxis_title='Close Price',
    yaxis2=dict(
        title='Momentum',
        titlefont=dict(color='purple'),
        tickfont=dict(color='purple'),
        overlaying='y',
        side='right'
    ),
    yaxis3=dict(
        title='RSI',
        titlefont=dict(color='orange'),
        tickfont=dict(color='orange'),
        overlaying='y',
        side='right',
        position=0.95
    )
)

# Hiển thị biểu đồ
fig.show()


In [20]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Mã chuẩn bị dữ liệu ở đây
# Giả sử 'data' là DataFrame của bạn và nó có 'Datetime' làm chỉ mục

# Tạo biểu đồ con: 3 hàng cho Giá đóng cửa, Momentum, và RSI
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Trading Signals with ARIMA, Momentum, and RSI', 'Momentum', 'RSI'),
                    vertical_spacing=0.1)

# Thêm giá đóng và dự đoán giá đóng vào hàng đầu tiên
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Dự Đoán Giá Đóng'), row=1, col=1)

# Thêm tín hiệu mua và bán trên cùng một biểu đồ con
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], mode='markers',
                         marker=dict(color='green', size=10, symbol='triangle-up'), name='Tín Hiệu Mua'), row=1, col=1)
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], mode='markers',
                         marker=dict(color='red', size=10, symbol='triangle-down'), name='Tín Hiệu Bán'), row=1, col=1)

# Thêm Momentum vào hàng thứ hai
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], mode='lines', name='Momentum'), row=2, col=1)

# Thêm RSI vào hàng thứ ba
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI'), row=3, col=1)

# Cập nhật layout
fig.update_layout(height=800, title_text="Chiến Lược Giao Dịch với Tín Hiệu Mua/Bán, Momentum và RSI")

# Hiển thị biểu đồ
fig.show()


# Day tin hieu qua Redis

In [4]:
data

,Open,High,Low,Close,Volume,Momentum,RSI,Predicted_Close,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,
2025-10-22 00:00:00,3873.04,3873.04,3862.16,3864.99,1188.6477,NaN,NaN,0.000000,False,False
2025-10-22 00:01:00,3864.98,3868.76,3862.67,3868.76,332.1784,3.77,NaN,3864.991882,False,False
2025-10-22 00:02:00,3868.75,3872.91,3868.65,3872.91,340.9633,4.15,NaN,3868.924053,False,False
2025-10-22 00:03:00,3872.91,3877.00,3872.90,3877.00,209.3692,4.09,NaN,3873.012368,False,False
2025-10-22 00:04:00,3877.00,3878.70,3874.81,3877.14,339.9579,0.14,NaN,3876.858788,False,False
...,...,...,...,...,...,...,...,...,...,...
2025-10-22 13:36:00,3836.81,3844.15,3836.81,3839.97,840.6678,3.17,33.675663,3836.630084,False,False
2025-10-22 13:37:00,3839.96,3851.83,3835.22,3847.25,994.9095,7.28,44.164429,3841.312692,False,False
2025-10-22 13:38:00,3847.25,3855.59,3847.25,3854.40,481.8656,7.15,50.917015,3846.867239,False,True


In [5]:
import pandas as pd
import plotly.graph_objects as go
import redis
import numpy as np
from datetime import datetime

# Nếu có tín hiệu thì đẩy qua Redis
# Datetime  Open    High    Low	Close   Volume  PredictClose Momentum RSI Buy_Signal  Sell_Signal
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=6) # Db0-5: Ck, Db6-10: Crypto, Db11-15: Forex

# Tạo hash key
hash_key = 'OG_CRTO_Arima, Momentum, RSI K13'
last_record = data.iloc[-2] # -2

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
    for field, value in last_record.to_dict().items():
        # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, (int, np.uint64)):
            value = str(value)
        r.hset(hash_key, field, value)  
        r.hset(hash_key, 'Symbol', symbol)
        r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
# Datetime  Open    High    Low	Close   Volume  PredictClose Momentum RSI Buy_Signal  Sell_Signal Symbol Insertdate
    print(last_record)   
else:
    print('Không có tín hiệu!')   

Open                   3854.35
High                   3859.63
Low                    3854.35
Close                  3858.38
Volume                 829.672
Momentum                  3.98
RSI                  53.944454
Predicted_Close    3855.042631
Buy_Signal               False
Sell_Signal               True
Name: 2025-10-22 13:39:00, dtype: object
